# Teacher → student knowledge distillation: VideoMAE → MoViNet-A0

The stronger VideoMAE-small teacher is fine-tuned first. Its softened logits then supervise a much smaller MoViNet-A0 student. Only the MoViNet student is needed at deployment.

In [ ]:
# Shoplifting detection — teacher → student knowledge distillation
!nvidia-smi -L
!pip -q install -U "transformers>=4.45" "accelerate>=0.34" "huggingface_hub>=0.25" "decord>=0.6.0" "scikit-learn>=1.4" "pandas>=2.0" "matplotlib>=3.8" "seaborn>=0.13" "tqdm>=4.66"

In [ ]:
import os, json, random, time, math, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score, confusion_matrix, classification_report, roc_curve, precision_recall_curve
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
import decord
decord.bridge.set_bridge("native")
assert torch.cuda.is_available(), "Enable a Colab GPU."
DEVICE=torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.benchmark=True
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
ROOT=Path("/content/shoplifting_project"); DATA=ROOT/"data"; CKPT=ROOT/"checkpoints"; FIG=ROOT/"figures"
for p in (DATA,CKPT,FIG): p.mkdir(parents=True,exist_ok=True)
UCF_REPO="jinmang2/ucf_crime"
TRAIN_LIST="UCF_Crimes-Train-Test-Split/Action_Recognition_splits/train_001.txt"
VAL_LIST="UCF_Crimes-Train-Test-Split/Action_Recognition_splits/test_001.txt"
STUDENT_ID="kfkas/movinet-a0-stream-pytorch"
LABELS=["normal","shoplifting"]; LABEL2ID={"normal":0,"shoplifting":1}; ID2LABEL={0:"normal",1:"shoplifting"}
FRAMES=16; IMG_SIZE=172; EPOCHS=3; LR=2e-4; WEIGHT_DECAY=1e-4
TEACHER_EPOCHS=2; TEACHER_LR=1e-5; TEMPERATURE=3.0; ALPHA=0.5; BETA=0.5
TEACHER_ID="MCG-NJU/videomae-small-finetuned-kinetics"
VRAM=torch.cuda.get_device_properties(0).total_memory/2**30
BATCH_SIZE=8 if VRAM>=20 else 4 if VRAM>=10 else 2
NUM_WORKERS=2
print(torch.cuda.get_device_name(0), f"{VRAM:.1f} GB VRAM; batch={BATCH_SIZE}")

In [ ]:
def list_items(path):
    local=hf_hub_download(repo_id=UCF_REPO,filename=path,repo_type="dataset")
    return [s.strip() for s in Path(local).read_text(errors="ignore").splitlines() if s.strip().endswith(".mp4")]
def select_split(list_path):
    items=list_items(list_path); pos=[]; neg=[]
    for rel in items:
        if rel.startswith("Shoplifting/"): pos.append((rel,1))
        elif "Normal_Videos" in rel: neg.append((rel,0))
    random.shuffle(pos); random.shuffle(neg)
    n=min(len(pos),len(neg))
    return pos[:n]+neg[:n]
train_items, val_items=select_split(TRAIN_LIST), select_split(VAL_LIST)
print("train:",len(train_items),"val:",len(val_items))
def ensure_download(items, split):
    out=[]
    for rel,y in tqdm(items,desc=f"download {split}"):
        p=hf_hub_download(repo_id=UCF_REPO,filename=rel,repo_type="dataset",local_dir=str(DATA/split))
        out.append((str(p),y,rel))
    return out
train_files=ensure_download(train_items,"train"); val_files=ensure_download(val_items,"val")
print("downloaded",len(train_files),len(val_files))

In [ ]:
def read_clip(path,n=FRAMES):
    vr=decord.VideoReader(path,ctx=decord.cpu(0))
    idx=np.linspace(0,len(vr)-1,n).astype(np.int64)
    return vr.get_batch(idx).asnumpy()
def prep(frames, size=IMG_SIZE, flip=False):
    x=torch.from_numpy(frames).permute(0,3,1,2).float()/255
    x=F.interpolate(x,size=(size,size),mode="bilinear",align_corners=False)
    if flip and random.random()<0.5: x=torch.flip(x,[-1])
    return x
class VideoDS(Dataset):
    def __init__(self,items,train=False): self.items=items; self.train=train
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        path,y,rel=self.items[i]
        try: x=prep(read_clip(path),flip=self.train)
        except Exception:
            x=torch.zeros(FRAMES,3,IMG_SIZE,IMG_SIZE); y=0
        return x,torch.tensor(y),rel
train_loader=DataLoader(VideoDS(train_files,True),batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
val_loader=DataLoader(VideoDS(val_files),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
print("batches",len(train_loader),len(val_loader))

In [ ]:
# Same files/loaders as the student, plus a teacher-specific 224x224 preprocessing path.
def prep_teacher(frames):
    x=torch.from_numpy(frames).permute(0,3,1,2).float()/255
    x=F.interpolate(x,size=(224,224),mode="bilinear",align_corners=False)
    mean=torch.tensor([0.485,0.456,0.406]).view(1,3,1,1); std=torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)
    return (x-mean)/std
class TeacherDS(VideoDS):
    def __getitem__(self,i):
        path,y,rel=self.items[i]
        try: x=prep_teacher(read_clip(path))
        except Exception: x=torch.zeros(FRAMES,3,224,224)
        return x,torch.tensor(y),rel
TEACHER_BATCH=2 if VRAM<20 else 4
teacher_train=DataLoader(TeacherDS(train_files,True),batch_size=TEACHER_BATCH,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
teacher_val=DataLoader(TeacherDS(val_files),batch_size=TEACHER_BATCH,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
print("teacher batch",TEACHER_BATCH)

### 1. Fine-tune the teacher
VideoMAE uses 16 frames at 224×224 with ImageNet normalization. The teacher is discarded after distillation.

In [ ]:
from transformers import AutoModelForVideoClassification
teacher=AutoModelForVideoClassification.from_pretrained(TEACHER_ID,num_labels=2,ignore_mismatched_sizes=True,label2id=LABEL2ID,id2label=ID2LABEL).to(DEVICE)
tscaler=torch.amp.GradScaler("cuda"); topt=torch.optim.AdamW(teacher.parameters(),lr=TEACHER_LR,weight_decay=1e-4)
best=-1; teacher_hist=[]
for epoch in range(1,TEACHER_EPOCHS+1):
    teacher.train(); total=0
    for x,y,_ in tqdm(teacher_train,desc=f"teacher epoch {epoch}"):
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE); topt.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=torch.float16): loss=F.cross_entropy(teacher(pixel_values=x).logits,y)
        tscaler.scale(loss).backward(); tscaler.unscale_(topt); torch.nn.utils.clip_grad_norm_(teacher.parameters(),1.0); tscaler.step(topt); tscaler.update(); total+=loss.item()
    teacher.eval(); ys=[]; ps=[]; probs=[]
    with torch.no_grad():
        for x,y,_ in teacher_val:
            x=x.to(DEVICE,non_blocking=True)
            with torch.autocast("cuda",dtype=torch.float16): o=teacher(pixel_values=x).logits
            ys+=y.tolist(); ps+=o.argmax(-1).cpu().tolist(); probs+=o.softmax(-1)[:,1].float().cpu().tolist()
    f1=precision_recall_fscore_support(ys,ps,average="binary",zero_division=0)[2]
    teacher_hist.append((epoch,total/len(teacher_train),f1)); print("loss",total/len(teacher_train),"val_f1",f1)
    if f1>best: best=f1; torch.save(teacher.state_dict(),CKPT/"videomae_teacher_best.pt")

### 2. Distill into MoViNet-A0
Loss = α·hard CE + β·temperature-scaled KL. Default: α=β=0.5, T=3.

In [ ]:
student=AutoModelForVideoClassification.from_pretrained(STUDENT_ID,trust_remote_code=True,num_labels=2,ignore_mismatched_sizes=True,label2id=LABEL2ID,id2label=ID2LABEL).to(DEVICE)
teacher.eval()
for p in teacher.parameters(): p.requires_grad=False
scaler=torch.amp.GradScaler("cuda"); opt=torch.optim.AdamW(student.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
def kd_loss(s,t,y,T=TEMPERATURE):
    hard=F.cross_entropy(s,y)
    soft=F.kl_div(F.log_softmax(s/T,-1),F.softmax(t/T,-1),reduction="batchmean")*(T*T)
    return ALPHA*hard+BETA*soft
best=-1; hist=[]
for epoch in range(1,EPOCHS+1):
    student.train(); total=0
    for x,y,_ in tqdm(train_loader,desc=f"distill epoch {epoch}"):
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE); opt.zero_grad(set_to_none=True)
        with torch.no_grad():
            xt=prep_teacher(read_clip(train_files[0][0])).unsqueeze(0).to(DEVICE) if False else None
        # Reuse the student batch and resize/normalize it for VideoMAE.
        xt=F.interpolate(x.flatten(0,1),size=(224,224),mode="bilinear",align_corners=False).view(x.size(0),x.size(1),3,224,224)
        mean=torch.tensor([0.485,0.456,0.406],device=DEVICE).view(1,1,3,1,1); std=torch.tensor([0.229,0.224,0.225],device=DEVICE).view(1,1,3,1,1); xt=(xt-mean)/std
        with torch.autocast("cuda",dtype=torch.float16):
            with torch.no_grad(): t=teacher(pixel_values=xt).logits
            s=student(pixel_values=x).logits
            loss=kd_loss(s,t,y)
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(student.parameters(),1.0); scaler.step(opt); scaler.update(); total+=loss.item()
    student.eval(); ys=[]; ps=[]; probs=[]
    with torch.no_grad():
        for x,y,_ in val_loader:
            x=x.to(DEVICE)
            with torch.autocast("cuda",dtype=torch.float16): o=student(pixel_values=x).logits
            ys+=y.tolist(); ps+=o.argmax(-1).cpu().tolist(); probs+=o.softmax(-1)[:,1].float().cpu().tolist()
    f1=precision_recall_fscore_support(ys,ps,average="binary",zero_division=0)[2]; hist.append((epoch,total/len(train_loader),f1)); print("loss",total/len(train_loader),"val_f1",f1)
    if f1>best: best=f1; torch.save({"state_dict":student.state_dict(),"history":hist},CKPT/"movinet_distilled_best.pt")

In [ ]:
student.eval(); ys=[]; ps=[]; probs=[]
with torch.no_grad():
    for x,y,_ in val_loader:
        x=x.to(DEVICE)
        with torch.autocast("cuda",dtype=torch.float16): o=student(pixel_values=x).logits
        ys+=y.tolist(); ps+=o.argmax(-1).cpu().tolist(); probs+=o.softmax(-1)[:,1].float().cpu().tolist()
m={"accuracy":accuracy_score(ys,ps),"balanced_accuracy":balanced_accuracy_score(ys,ps)}
m["precision"],m["recall"],m["f1"],_=precision_recall_fscore_support(ys,ps,average="binary",zero_division=0)
m["roc_auc"]=roc_auc_score(ys,probs); m["average_precision"]=average_precision_score(ys,probs)
print(json.dumps(m,indent=2))
plt.figure(figsize=(5,4)); sns.heatmap(confusion_matrix(ys,ps),annot=True,fmt="d",xticklabels=LABELS,yticklabels=LABELS); plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Distilled student confusion matrix"); plt.show()
fpr,tpr,_=roc_curve(ys,probs); pr,re,_=precision_recall_curve(ys,probs)
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].plot(fpr,tpr); ax[0].set_title(f"ROC AUC={m['roc_auc']:.3f}"); ax[1].plot(re,pr); ax[1].set_title(f"PR AP={m['average_precision']:.3f}"); plt.show()
print(classification_report(ys,ps,target_names=LABELS,digits=4))

In [ ]:
# Optional comparison against the direct-fine-tuning checkpoint from notebook 01.
direct_path=CKPT/"movinet_direct_best.pt"
if direct_path.exists():
    direct=AutoModelForVideoClassification.from_pretrained(STUDENT_ID,trust_remote_code=True,num_labels=2,ignore_mismatched_sizes=True,label2id=LABEL2ID,id2label=ID2LABEL).to(DEVICE)
    direct.load_state_dict(torch.load(direct_path,map_location=DEVICE)["state_dict"]); direct.eval()
    def quick(m):
        y=[];p=[]
        with torch.no_grad():
            for x,t,_ in val_loader:
                x=x.to(DEVICE)
                with torch.autocast("cuda",dtype=torch.float16): q=m(pixel_values=x).logits
                y+=t.tolist();p+=q.argmax(-1).cpu().tolist()
        return precision_recall_fscore_support(y,p,average="binary",zero_division=0)[2]
    print({"direct_f1":quick(direct),"distilled_f1":m["f1"]})
else: print("Run notebook 01 first to compare checkpoints.")

### Deployment note
Use `movinet_distilled_best.pt` only. The VideoMAE teacher is a training-time dependency, not a runtime dependency. For a smoke test, reduce both epoch counts and slice the downloaded file lists before creating DataLoaders.

Dataset/model references: UCF-Crime mirror `jinmang2/ucf_crime`; teacher `MCG-NJU/videomae-small-finetuned-kinetics`; student `kfkas/movinet-a0-stream-pytorch`.